In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import open3d as o3d
import yaml

# 导入你项目里的超级包装盒
from environment import ReKepOGEnv


with open('/home/clt/Rekep/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

env_config = full_config['env']
scene_file = "/home/clt/Rekep/configs/og_scene_file_red_pen.json"
env = ReKepOGEnv(config=env_config, scene_file=scene_file, verbose=True)

In [ ]:
env.reset()

In [ ]:
obs_dict = env.get_cam_obs()
data = obs_dict[0]

In [ ]:
from keypoint_proposal import KeypointProposer

# 初始化这个加工厂 (传入 keypoint_proposer 的相关配置)
o_KeypointProposer = KeypointProposer(full_config['keypoint_proposer'])

# 准备原材料 (从上一步 env 获取的 data 中提取)
raw_rgb = torch.as_tensor(data['rgb'])
raw_points = torch.as_tensor(data['points'])
raw_mask = torch.as_tensor(data['seg'])

print(f"数据类型已统一为: {raw_rgb.dtype}, 形状: {raw_rgb.shape}")

# 进入 _preprocess 工厂流水线

transformed_rgb, rgb, points, masks_list, shape_info = o_KeypointProposer._preprocess(
    raw_rgb, raw_points, raw_mask
)

print(f"--- 预处理流水线完成 ---")
print(f"拆分后的 Mask 图层数量: {len(masks_list)} (现在它是多层二值化掩码了)")
print(f"适配 DINOv2 的图片尺寸: {transformed_rgb.shape}")


In [ ]:
import matplotlib.pyplot as plt

# 1. 准备原材料
raw_rgb = torch.as_tensor(data['rgb'])
raw_points = torch.as_tensor(data['points'])
raw_mask = torch.as_tensor(data['seg'])

# 2. 一键启动超级流水线 (代替你刚才写的所有手动步骤！)
candidate_keypoints, projected_img = o_KeypointProposer.get_keypoints(raw_rgb, raw_points, raw_mask)

# 3. 直接看毕业照
plt.figure(figsize=(10, 10))
plt.imshow(projected_img)
plt.title("Generated Keypoints (Candidates for VLM)", fontsize=16)
plt.axis('off')
plt.show()

In [ ]:
#以上是我们获取静态候选点的流程，但是在实际中，摄像机画面是会移动的，我们需要注册我们的候选点，把它与实际物体相绑定
#让我们回到environment.py，聚焦register_keypoints()函数
'''
    还记得我们在learn1中第五个代码是如何计算仿真物体的坐标吗？——网格化、采样求坐标
    这里也要先网格化仿真物体（不如说本来就是网格mesh），然后在物体的表皮上随机撒下 1000 个点，复杂的连续曲面就变成了
    1000 个离散的(X,Y,Z) 坐标，拿着之前 K-Means 算出来的那个“虚拟关键点”，去挨个测量它和这 1000 粒芝麻的欧氏距离（直线距离）。找出距离最短的那一粒芝进行登记register
'''
'''
    输入是上一步得到的候选点的3d坐标
    输出是  贴在物体表皮上的物理 3D 坐标（注册点具体坐标）、对应的mesh具体结构（注册点所在结构）、以及所属权（属于哪个物体）记录

'''


# ==========================================
# 1.跨越虚实边界：将数学点绑定到物理引擎上
# ==========================================
print("\n[系统提示] 开始将数学坐标吸附到物理实体上...")

# 核心魔法：给这些点打上物理钢印！
env.register_keypoints(candidate_keypoints)

print("[系统提示] 所有关键点已成功打上 GPS 追踪钢印！")

# 让我们来查一下 0 号和 1 号关键点现在的“户口本”
try:
    obj_0 = env.get_object_by_keypoint(0)
    print(f"✅ 0号关键点目前牢牢绑定在物体: {obj_0.name} 的表皮上")

    # 提取最新的实时物理坐标（以后不管物体怎么滚，都调用这句）
    live_positions = env.get_keypoint_positions()
    print(f"📍 0号关键点当前的绝对 3D 坐标是: {live_positions[0]}")
except Exception as e:
    print(f"提取失败，可能需要检查: {e}")

In [ ]:
#2.操控本体的接口函数
#操控机器人运动的封装好的函数，在此先省略
